In [1]:
import requests
import numpy as np
import pandas as pd

CITY = "Sargodha"
LAT, LON = 32.0836, 72.6711

def nasa_daily(lat, lon, start="20010101", end="20261119"):
    url = "https://power.larc.nasa.gov/api/temporal/daily/point"
    params = {
        "parameters": "ALLSKY_SFC_SW_DWN",
        "community": "RE",
        "latitude": lat,
        "longitude": lon,
        "start": start,
        "end": end,
        "format": "JSON",
    }
    data = requests.get(url, params=params, timeout=120).json()
    s = pd.Series(data["properties"]["parameter"]["ALLSKY_SFC_SW_DWN"])
    s.index = pd.to_datetime(s.index, format="%Y%m%d")
    return s.replace(-999, np.nan)

daily = nasa_daily(LAT, LON)
print("Total days:", len(daily), "| Missing:", daily.isna().sum())
print("Last valid day:", daily.last_valid_index().date())


by_month = daily.resample("MS").agg(["mean", "count"])
monthly = by_month.loc[by_month["count"] >= 25, "mean"]

PR = 0.80  
kwh_per_kw = monthly * monthly.index.days_in_month * PR

print("\nLast 3 completed months (kWh/kW):")
print(kwh_per_kw.tail(3).round(1))
print("\nAverage annual yield (2001-2025):", round(kwh_per_kw.loc["2001":"2025"].groupby(lambda d: d.year).sum().mean(), 0), "kWh/kW/year")

Total days: 9454 | Missing: 5
Last valid day: 2026-11-14

Last 3 completed months (kWh/kW):
2026-08-01    136.2
2026-09-01    124.1
2026-10-01    102.5
Freq: MS, dtype: float64

Average annual yield (2001-2025): 1497.0 kWh/kW/year


In [2]:
t = kwh_per_kw.rename("y").to_frame()
t["month"] = t.index.month

# Base lags safe for up to 4 months ahead (Feb 2027 uses Oct 2026 as lag4 because Nov is Incomplete)
t["lag4"] = t["y"].shift(4)
t["lag12"] = t["y"].shift(12)
t["lag24"] = t["y"].shift(24)

# Leakage-free expanding climatology (average of this calendar month across all prior years)
past_mean = lambda v: v.shift(1).expanding().mean()
t["clim"] = t.groupby("month")["y"].transform(past_mean)

# --- Added Feature (Chapter 4): Cyclical Month Encoding ---
# Tells tree/linear models that December (12) and January (1) are adjacent in the seasonal cycle
t["sin_month"] = np.sin(2 * np.pi * t["month"] / 12)
t["cos_month"] = np.cos(2 * np.pi * t["month"] / 12)

t = t.dropna()

BASE_FEATS = ["month", "lag4", "lag12", "lag24", "clim"]
MOD_FEATS = BASE_FEATS + ["sin_month", "cos_month"]

print("Clean table shape:", t.shape)
print("\nLast 3 rows of feature table:")
print(t.tail(3).round(2))

Clean table shape: (286, 8)

Last 3 rows of feature table:
                 y  month    lag4   lag12   lag24    clim  sin_month  \
2026-08-01  136.16      8  138.86  133.03  118.84  140.88      -0.87   
2026-09-01  124.13      9  172.45  130.27  133.58  132.11      -1.00   
2026-10-01  102.51     10  168.85  117.43  120.28  119.17      -0.87   

            cos_month  
2026-08-01       -0.5  
2026-09-01       -0.0  
2026-10-01        0.5  


In [3]:
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error

splits = TimeSeriesSplit(n_splits=5, test_size=12)

scores = {
    "seasonal naive": [],
    "climatology": [],
    "XGBoost (base)": [],
    "XGBoost (+cyclical)": [],
}

for train_idx, test_idx in splits.split(t):
    train, test = t.iloc[train_idx], t.iloc[test_idx]

    # 1. Seasonal Naive (same month last year)
    scores["seasonal naive"].append(mean_absolute_error(test["y"], test["lag12"]))

    # 2. Climatology
    scores["climatology"].append(mean_absolute_error(test["y"], test["clim"]))

    # 3. Base XGBoost (5 features: month, lag4, lag12, lag24, clim)
    m_base = xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=42)
    m_base.fit(train[BASE_FEATS], train["y"])
    scores["XGBoost (base)"].append(mean_absolute_error(test["y"], m_base.predict(test[BASE_FEATS])))

    # 4. Modified XGBoost (7 features: adds sin_month & cos_month)
    m_mod = xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=42)
    m_mod.fit(train[MOD_FEATS], train["y"])
    scores["XGBoost (+cyclical)"].append(mean_absolute_error(test["y"], m_mod.predict(test[MOD_FEATS])))

    print(test.index[0].strftime("%Y-%m"), "to", test.index[-1].strftime("%Y-%m"))

print("\n--- 5-Year Walk-Forward Results (Sargodha) ---")
mean_y = t["y"].mean()
for name, s in scores.items():
    mae = np.mean(s)
    pct = (mae / mean_y) * 100
    print(f"{name:22s} MAE: {mae:5.2f} kWh/kW/month  (~{pct:.1f}% error)")

2021-11 to 2022-10
2022-11 to 2023-10
2023-11 to 2024-10
2024-11 to 2025-10
2025-11 to 2026-10

--- 5-Year Walk-Forward Results (Sargodha) ---
seasonal naive         MAE: 11.05 kWh/kW/month  (~8.9% error)
climatology            MAE:  7.87 kWh/kW/month  (~6.3% error)
XGBoost (base)         MAE:  7.87 kWh/kW/month  (~6.3% error)
XGBoost (+cyclical)    MAE:  7.89 kWh/kW/month  (~6.3% error)


In [4]:
from datetime import date

#Climatology Forecast: Full Historical Climatology (all completed months through Oct 2026)
full_clim = kwh_per_kw.groupby(kwh_per_kw.index.month).mean()

forecast_months = [
    ("2026-12", 12),
    ("2027-01", 1),
    ("2027-02", 2),
]

forecast_df = pd.DataFrame([
    {
        "Month": label,
        "Forecast (kWh/kW)": round(full_clim[m], 2),
        "Method": "Climatology (2001-2026 mean)",
        "Generated Date": str(date.today()),
    }
    for label, m in forecast_months
])

print("=== SARGODHA 3-MONTH SOLAR FORECAST ===")
print(forecast_df.to_string(index=False))

system_kw = 10.0
annual_yield_per_kw = 1496.9
annual_kwh = system_kw * annual_yield_per_kw 

system_price_pkr = 1_250_000   
import_tariff_pkr = 65.0
export_rate_pkr = 32.0         # NEPRA/FESCO net-metering export-rate
self_consumed_share = 0.55     

self_consumed_kwh = annual_kwh * self_consumed_share          # 8,233.0 kWh
exported_kwh = annual_kwh * (1 - self_consumed_share)         # 6,736.0 kWh

annual_savings_pkr = (self_consumed_kwh * import_tariff_pkr) + (exported_kwh * export_rate_pkr)
payback_years = system_price_pkr / annual_savings_pkr

print(f"Self-consumed savings: PKR {self_consumed_kwh * import_tariff_pkr:,.0f}/year")
print(f"Export credit savings: PKR {exported_kwh * export_rate_pkr:,.0f}/year")
print(f"Total annual savings:  PKR {annual_savings_pkr:,.0f}/year")
print(f"Payback period:        {payback_years:.2f} years")

=== SARGODHA 3-MONTH SOLAR FORECAST ===
  Month  Forecast (kWh/kW)                       Method Generated Date
2026-12              75.69 Climatology (2001-2026 mean)     2026-11-19
2027-01              75.53 Climatology (2001-2026 mean)     2026-11-19
2027-02              88.49 Climatology (2001-2026 mean)     2026-11-19

=== SARGODHA 10 kW SOLAR PAYBACK SUMMARY ===
Annual Yield:      1,496.9 kWh/kW/year
Annual Generation: 14,969 kWh/year
Installed Cost:    PKR 1,250,000
Annual Savings:    PKR 748,450/year (@ PKR 50/kWh)
Simple Payback:    1.67 years


Here are the answers to your couple of questions mentioned in the assignment.
* Model is only worth training and utilizing if it beats a baseline model. XGboost was unable to beat climatology. Even with the added features. Random split is wrong in such scenarios because it is risky as the model may overlook into the future (target prediction value).
* Climatology won on the basis of MAE. XGboost tied with Climatology and we went further with climatology to have zero risk of overfitting.
